# Haplogroup dispersion (internal analysis template)

Use only an authorized, provenance-aware query to prepare the input CSV. This notebook handles assignment metadata, not sequence data. Its results are internal by default. Before any public release, validate every contributing sample with the trusted publication gate and apply the governance-configured minimum cohort size. Preserve this notebook's revision, environment, parameters, inputs, and generated Media_Asset links with the analysis record.

In [ ]:
from pathlib import Path
import csv
from uuid import UUID

from genomics.haplogroups import (
    HaplogroupObservation,
    aggregate_haplogroup_frequencies,
    frequency_chart_spec,
)

ASSIGNMENTS_CSV = Path("inputs/haplogroup_assignments.csv")
ANALYSIS_PARAMETERS = {
    "input_file": str(ASSIGNMENTS_CSV),
    "frequency_unit": "assigned sample",
    "public_release": False,
}

In [ ]:
required_columns = {
    "bio_sample_id", "region_id", "time_slice", "marker_type",
    "haplogroup_label", "confidence_lower", "confidence_upper",
}
with ASSIGNMENTS_CSV.open(newline="", encoding="utf-8") as source:
    reader = csv.DictReader(source)
    if not required_columns.issubset(reader.fieldnames or ()):
        raise ValueError(f"Input CSV must include: {sorted(required_columns)}")
    observations = tuple(
        HaplogroupObservation(
            bio_sample_id=UUID(row["bio_sample_id"]),
            region_id=UUID(row["region_id"]),
            time_slice=row["time_slice"],
            marker_type=row["marker_type"],
            haplogroup_label=row["haplogroup_label"],
            confidence_lower=float(row["confidence_lower"]),
            confidence_upper=float(row["confidence_upper"]),
        )
        for row in reader
    )
if not observations:
    raise ValueError("Input CSV contains no haplogroup assignments")

report = aggregate_haplogroup_frequencies(observations)
frequency_rows = [row.to_dict() for row in report.rows]
chart_spec = frequency_chart_spec(report)
frequency_rows

The table reports observed sample proportions, not population estimates or inferential confidence intervals. The lower/upper confidence values summarize the stored assignment intervals. Do not treat missing assignments as negative evidence. Regional and temporal grouping must be sourced and versioned upstream.